## tl;dr

Sunday 2 August 2026 stayed below the kitchen pacing cap. The busiest scheduled 30-minute arrival window held 10 food covers against a limit of 15, leaving 5 covers spare.

## Context & Methods

This notebook reads live `system_settings`, `special_hours`, and anonymised `table_bookings` fields from Supabase. It mirrors the application's centred, half-open pacing window: `[slot - 15 minutes, slot + 15 minutes)`.

### Key Assumptions

- The date is interpreted in Europe/London.
- Kitchen pacing counts food bookings only.
- Cancelled bookings are excluded. No-shows are shown separately because they occupied diary capacity until marked.
- `committed_party_size` is preferred over `party_size`, matching the application.

In [1]:
from pathlib import Path
import os
import requests
import pandas as pd
from dotenv import load_dotenv

ANALYSIS_DATE = '2026-08-02'
repo = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '.env.local').exists())
load_dotenv(repo / '.env.local')
base_url = os.environ['NEXT_PUBLIC_SUPABASE_URL'].rstrip('/')
service_key = os.environ['SUPABASE_SERVICE_ROLE_KEY']
headers = {'apikey': service_key, 'Authorization': f'Bearer {service_key}'}

def fetch(table, **params):
    response = requests.get(f'{base_url}/rest/v1/{table}', headers=headers, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

## Data

In [2]:
setting_keys = [
    'kitchen_pacing_enabled',
    'kitchen_pacing_window_minutes',
    'kitchen_pace_covers_regular',
    'kitchen_pace_covers_sunday',
    'kitchen_walk_in_reserve_regular',
    'kitchen_walk_in_reserve_sunday',
]
settings_rows = fetch(
    'system_settings',
    select='key,value,updated_at',
    key=f"in.({','.join(setting_keys)})",
)
special_rows = fetch(
    'special_hours',
    select='date,kitchen_pace_covers,kitchen_walk_in_reserve',
    date=f'eq.{ANALYSIS_DATE}',
)
booking_rows = fetch(
    'table_bookings',
    select='booking_time,party_size,committed_party_size,booking_purpose,status,source,created_at,seated_at,left_at,no_show_at,cancelled_at',
    booking_date=f'eq.{ANALYSIS_DATE}',
    order='booking_time.asc',
)
len(booking_rows), special_rows

(19, [])

In [3]:
settings = {row['key']: row['value']['value'] for row in settings_rows}
override = special_rows[0] if special_rows else {}
pace = override.get('kitchen_pace_covers') or settings['kitchen_pace_covers_sunday']
reserve = override.get('kitchen_walk_in_reserve')
if reserve is None:
    reserve = settings['kitchen_walk_in_reserve_sunday']
ceiling = max(0, pace - reserve)
{'enabled': settings['kitchen_pacing_enabled'], 'window_minutes': settings['kitchen_pacing_window_minutes'], 'pace': pace, 'reserve': reserve, 'online_ceiling': ceiling, 'date_override': bool(special_rows)}

{'enabled': True,
 'window_minutes': 30,
 'pace': 15,
 'reserve': 0,
 'online_ceiling': 15,
 'date_override': False}

In [4]:
bookings = pd.DataFrame(booking_rows)
bookings['covers'] = bookings['committed_party_size'].fillna(bookings['party_size']).astype(int)
bookings['minute'] = pd.to_timedelta(bookings['booking_time']).dt.total_seconds().div(60).astype(int)
food = bookings.loc[bookings['booking_purpose'].eq('food')].copy()
food['cancelled'] = food['status'].eq('cancelled')
food['no_show'] = food['status'].eq('no_show')
food['attended'] = food['seated_at'].notna() & ~food['cancelled'] & ~food['no_show']
pd.DataFrame({
    'covers': [
        int(food.loc[food['attended'], 'covers'].sum()),
        int(food.loc[food['no_show'], 'covers'].sum()),
        int(food.loc[food['cancelled'], 'covers'].sum()),
    ]
}, index=['attended food covers', 'no-show food covers', 'cancelled food covers'])

,covers
attended food covers,27
no-show food covers,11
cancelled food covers,8


## Results

In [5]:
window_minutes = int(settings['kitchen_pacing_window_minutes'])
half_window = window_minutes / 2
centres = range(12 * 60, 18 * 60 + 1, 15)
results = []
for centre in centres:
    in_window = food['minute'].ge(centre - half_window) & food['minute'].lt(centre + half_window)
    booked_covers = int(food.loc[in_window & ~food['cancelled'], 'covers'].sum())
    attended_covers = int(food.loc[in_window & food['attended'], 'covers'].sum())
    results.append({
        'centre': f'{centre // 60:02d}:{centre % 60:02d}',
        'window': f'{int(centre-half_window)//60:02d}:{int(centre-half_window)%60:02d}-{int(centre+half_window)//60:02d}:{int(centre+half_window)%60:02d}',
        'booked_food_covers': booked_covers,
        'attended_food_covers': attended_covers,
        'spare_vs_cap': ceiling - booked_covers,
    })
pacing = pd.DataFrame(results)
pacing.loc[pacing['booked_food_covers'].gt(0)].sort_values(['booked_food_covers', 'centre'], ascending=[False, True]).head(10)

,centre,window,booked_food_covers,attended_food_covers,spare_vs_cap
6,13:30,13:15-13:45,10,10,5
5,13:15,13:00-13:30,8,6,7
7,13:45,13:30-14:00,8,8,7
8,14:00,13:45-14:15,7,7,8
9,14:15,14:00-14:30,7,7,8
12,15:00,14:45-15:15,7,2,8
4,13:00,12:45-13:15,6,4,9
13,15:15,15:00-15:30,5,0,10
16,16:00,15:45-16:15,4,4,11
17,16:15,16:00-16:30,4,4,11


In [6]:
peak = pacing.loc[pacing['booked_food_covers'].idxmax()]
assert int(peak['booked_food_covers']) <= ceiling
assert int(food.loc[food['attended'], 'covers'].sum()) == 27
assert int(food.loc[food['no_show'], 'covers'].sum()) == 11
assert int(food.loc[food['cancelled'], 'covers'].sum()) == 8
{
    'peak_window': peak['window'],
    'peak_booked_food_covers': int(peak['booked_food_covers']),
    'cap': ceiling,
    'spare_capacity': int(peak['spare_vs_cap']),
    'cap_used_percent': round(100 * int(peak['booked_food_covers']) / ceiling, 1),
}

{'peak_window': '13:15-13:45',
 'peak_booked_food_covers': 10,
 'cap': 15,
 'spare_capacity': 5,
 'cap_used_percent': 66.7}

## Takeaways

- Peak scheduled kitchen pacing was 10 covers in the 13:15-13:45 window: 67% of the 15-cover cap.
- The day had 27 attended food covers in total.
- Another 11 food covers were no-shows and 8 were cancelled; these outcomes affect a retrospective view of demand but did not push any pacing window to the cap.
- There was no date-specific pacing override.